<a href="https://colab.research.google.com/github/Crayon897/miami-dade-311-analysis/blob/main/311_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Which Miami-Dade ZIP Codes Wait Longest for County Services?

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/311_2023.csv')
df.head()

Mounted at /content/drive


,ticket_id,issue_type,issue_description,case_owner,case_owner_description,street_address,city,state,zip_code,neighborhood_district,...,sr_xcoordinate,sr_ycoordinate,latitude,longitude,method_received,sr_priority,goal_days,actual_completed_days,ticket_year,ObjectId
0,23-10000374,BULKY TRASH REQUEST BY APPOINTMENT,NaN,Solid_Waste_Management,Solid Waste Management,21122 SW 92ND CT,Town_of_Cutler_Bay,Florida,33189.0,District 8,...,873554.351,450424.639,25.571308,-80.340525,WEB,STANDARD,8.0,11.0,2023,1
1,23-10000376,BREEDER / PET STORE FOLLOW-UP,NaN,Enforcement_Section-3-36,Enforcement Section,9690 NW 41ST ST,City_of_Doral,Florida,33178.0,District 12,...,869217.668,537584.781,25.811171,-80.352385,INHOUSE,STANDARD,180.0,1.0,2023,2
2,23-10000377,INJURED ANIMAL,NaN,Enforcement_Section-3-36,Enforcement Section,NW 142ND LN & NW 87TH AVE,City_of_Miami_Lakes,Florida,33018.0,District 13,...,872951.375,570986.622,25.903017,-80.340527,INHOUSE,EMERGNCY,180.0,0.0,2023,3
3,23-10000378,BULKY TRASH REQUEST BY APPOINTMENT,NaN,Solid_Waste_Management,Solid Waste Management,10380 SW 66TH TER,Miami_Dade_County,Florida,33173.0,District 10,...,866149.750,499350.780,25.706020,-80.362272,WEB,STANDARD,8.0,28.0,2023,4
4,23-10000379,ILLEGAL DUMPING / LITTER,NaN,COM_SW,COM Solid Waste,1656 SW 6TH ST,City_of_Miami,Florida,33135.0,District 5,...,912160.747,521835.219,25.767201,-80.222142,IPHONE,STANDARD,7.0,16.0,2023,5


In [ ]:
df.info()

In [ ]:
df['issue_type'].value_counts().head(15)

In [ ]:
df['ticket_status'].value_counts()

In [ ]:
df['created'] = pd.to_datetime(df['ticket_created_date_time'], errors='coerce', utc=True)
df['closed'] = pd.to_datetime(df['ticket_closed_date_time'], errors='coerce', utc=True)
print('Created dates that failed:', df['created'].isna().sum())
print('Closed dates missing:', df['closed'].isna().sum())

df['response_days'] = (df['closed'] - df['created']).dt.total_seconds() / 86400
df['response_days'].describe()

In [ ]:
start = len(df)

clean = df.dropna(subset=['created', 'closed', 'zip_code'])
print('Removed for missing dates or ZIP:', start - len(clean))

before = len(clean)
clean = clean[clean['response_days'] >= 0].copy()
print('Removed for closing before opening:', before - len(clean))

before = len(clean)
clean = clean[clean['ticket_status'] == 'CLOSED'].copy()
print('Removed for not being truly closed:', before - len(clean))

clean['zip_code'] = clean['zip_code'].astype(int).astype(str)
print('Rows left:', len(clean))

In [ ]:
for t in ['POTHOLE', 'ILLEGAL DUMPING - WM', 'ILLEGAL DUMPING / LITTER']:
    s = clean[clean['issue_type'] == t]
    print(t, ':', len(s), 'requests across', s['zip_code'].nunique(), 'ZIP codes')

In [ ]:
ISSUE = 'POTHOLE'

subset = clean[clean['issue_type'] == ISSUE]
print(ISSUE, ':', len(subset), 'requests')
print('Median days to close:', round(subset['response_days'].median(), 1))

In [ ]:
subset[['response_days', 'actual_completed_days', 'goal_days']].head(10)

In [ ]:
df[df['issue_type'] == 'POTHOLE']['ticket_status'].value_counts()

In [ ]:
subset = subset.copy()
subset['met_goal'] = subset['response_days'] <= subset['goal_days']

by_zip = subset.groupby('zip_code').agg(
    requests=('response_days', 'count'),
    median_days=('response_days', 'median'),
    pct_met_goal=('met_goal', 'mean'),
)
by_zip['median_days'] = by_zip['median_days'].round(1)
by_zip['pct_met_goal'] = (by_zip['pct_met_goal'] * 100).round()

print('ZIP codes:', len(by_zip))
by_zip = by_zip[by_zip['requests'] >= 30]
print('ZIP codes with 30+ requests:', len(by_zip))

by_zip = by_zip.sort_values('median_days', ascending=False)
by_zip.head(10)

In [ ]:
by_zip.tail(10)

In [ ]:
print('Goal days the county uses for potholes:', subset['goal_days'].unique())
print('Countywide median:', round(subset['response_days'].median(), 1), 'days')
print('Fixed within goal:', round(subset['met_goal'].mean() * 100), '%')
print()
print('Slowest ZIP:', by_zip.index[0], '-', by_zip['median_days'].iloc[0], 'days')
print('Fastest ZIP:', by_zip.index[-1], '-', by_zip['median_days'].iloc[-1], 'days')
print('Slowest is', round(by_zip['median_days'].iloc[0] / by_zip['median_days'].iloc[-1], 1), 'times the fastest')

In [ ]:
pot = df[(df['issue_type'] == 'POTHOLE') & (df['ticket_status'] == 'CLOSED')]
print('Closed potholes:', len(pot))
print('Missing ZIP:', pot['zip_code'].isna().sum())
print('Missing close date:', pot['closed'].isna().sum())
print('Closed before opened:', (pot['response_days'] < 0).sum())

In [ ]:
import matplotlib.pyplot as plt

med = subset['response_days'].median()

plt.figure(figsize=(8, 4.5))
plt.hist(subset['response_days'].clip(upper=30), bins=30, color='steelblue', edgecolor='white')
plt.axvline(3, color='red', linestyle='--', label='County goal: 3 days')
plt.axvline(med, color='black', label=f'Median: {med:.1f} days')
plt.title("Most potholes miss the county's 3-day goal")
plt.xlabel('Days to close (anything over 30 is counted at 30)')
plt.ylabel('Number of potholes')
plt.legend()
plt.savefig('chart1_response_times.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
med = subset['response_days'].median()
chart = pd.concat([by_zip.head(10), by_zip.tail(10)]).sort_values('median_days')
colors = ['indianred' if d > med else 'steelblue' for d in chart['median_days']]

plt.figure(figsize=(8, 7))
plt.barh(chart.index, chart['median_days'], color=colors)
plt.axvline(med, color='black', linestyle='--', label=f'Countywide median: {med:.1f} days')
plt.title('The slowest ZIP codes wait up to twice the county median for pothole repairs')
plt.xlabel('Median days to close')
plt.ylabel('ZIP code')
plt.legend()
plt.savefig('chart2_zip_codes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
med = subset['response_days'].median()
r = by_zip['requests'].corr(by_zip['median_days'])
print('Correlation between requests and median days:', round(r, 2))

plt.figure(figsize=(8, 5))
plt.scatter(by_zip['requests'], by_zip['median_days'], color='steelblue')
plt.axhline(med, color='black', linestyle='--', label=f'Countywide median: {med:.1f} days')
for z in [by_zip.index[0], by_zip.index[-1], by_zip['requests'].idxmax()]:
    plt.annotate(z, (by_zip.loc[z, 'requests'], by_zip.loc[z, 'median_days']),
                 xytext=(5, 5), textcoords='offset points')
plt.title("Busier ZIP codes don't wait longer for pothole repairs")
plt.xlabel('Pothole requests in 2023')
plt.ylabel('Median days to close')
plt.legend()
plt.savefig('chart3_busy_vs_slow.png', dpi=150, bbox_inches='tight')
plt.show()